# Model 1 (Baseline): Logistic Regression, Single Feature

This model is intended to be a baseline for future analysis and a basis for comparison. 

In [1]:
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder \
    .appName("Final_Project_Model1") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/apps/software/standard/core/pyspark/4.2.0-py3.14/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/26 17:04:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
DATA_FILEPATH = '/standard/ds7200-apt4c/team2/ptb_superclass_v4'

In [30]:
df = spark.read.parquet(DATA_FILEPATH)
df.select("ecg_id").show(5)

+------+
|ecg_id|
+------+
|   217|
|   290|
|   441|
|   733|
|   800|
+------+
only showing top 5 rows


Selected the age field since is makes intuitive sense that older adults are more likely to have a myocardial infarction. 

In [31]:
from pyspark.sql.functions import asc
from pyspark.sql import functions as F

df.createOrReplaceTempView("m1")

m1_query1 = spark.sql("SELECT MI, AVG(age), COUNT(ecg_id) \
                          FROM m1 \
                          GROUP BY MI \
                          ORDER BY MI asc")
m1_query1.show(20)

+---+-----------------+-------------+
| MI|         avg(age)|count(ecg_id)|
+---+-----------------+-------------+
|  0|57.40799396681749|        15912|
|  1|66.77954628613246|         5466|
+---+-----------------+-------------+



#### Train/Validate/Test Split

In [33]:
train = df.filter("strat_fold <= 8")
validate = df.filter("strat_fold == 9")
test = df.filter("strat_fold == 10")

#### Packaging variables into a feature column

In [34]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=["age"],
                           outputCol="features")

tr = assembler.transform(train)
te = assembler.transform(test)

#### Standardizing the features

In [38]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")
scalerModel = scaler.fit(tr) 

scaledTrain = scalerModel.transform(tr) 
scaledTest = scalerModel.transform(te)

scaledTrain.select("MI","features","scaledFeatures").show(5, truncate=False)

+---+--------+--------------------+
|MI |features|scaledFeatures      |
+---+--------+--------------------+
|1  |[73.0]  |[4.317112084502453] |
|0  |[70.0]  |[4.139696519385914] |
|0  |[83.0]  |[4.908497301557583] |
|0  |[67.0]  |[3.9622809542693744]|
|0  |[65.0]  |[3.844003910858348] |
+---+--------+--------------------+
only showing top 5 rows


#### Fitting the model

In [43]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(labelCol='MI',
                        featuresCol='scaledFeatures',
                        maxIter=10,
                        regParam=0,
                        elasticNetParam=0)

lrModel = lr.fit(scaledTrain)

print("Coefficients: " + str(lrModel.coefficients))
print("Intercept: " + str(lrModel.intercept))

Coefficients: [0.5733199000950826]
Intercept: -3.1646225877909258


ROC Curve

In [44]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

lrPred = lrModel.transform(scaledTest)
lrPred.select("MI", "rawPrediction",'probability','prediction').show(5,truncate=False)

evaluator = BinaryClassificationEvaluator(
    rawPredictionCol="rawPrediction",
    labelCol="MI",
    metricName="areaUnderROC"
)

roc_auc = evaluator.evaluate(lrPred)

print("Area under ROC:", roc_auc)

+---+----------------------------------------+----------------------------------------+----------+
|MI |rawPrediction                           |probability                             |prediction|
+---+----------------------------------------+----------------------------------------+----------+
|1  |[1.0624945237204377,-1.0624945237204377]|[0.7431669633573353,0.25683303664266466]|0.0       |
|1  |[0.5200098620248279,-0.5200098620248279]|[0.6271500723815492,0.3728499276184508] |0.0       |
|1  |[0.6556310274487305,-0.6556310274487305]|[0.6582782775764694,0.3417217224235306] |0.0       |
|0  |[1.367642145924218,-1.367642145924218]  |[0.7969989396481634,0.20300106035183663]|0.0       |
|1  |[0.5878204447367787,-0.5878204447367787]|[0.6428648983940005,0.3571351016059995] |0.0       |
+---+----------------------------------------+----------------------------------------+----------+
only showing top 5 rows
Area under ROC: 0.7085986718689674
